[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/vortices/vortices.ipynb)

# Leapfrogging vortices

Two pairs of vortices in a plane of fluid travel side by side and take turns passing through each other. The velocity gradient at each point is a map from a small step to the change in velocity, and contracting it against an open vector gives the derivative of the flow: where the fluid spreads, and how fast it turns.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from dataclasses import replace

import numpy as np

from numga import NumpyContext
from examples.mechanics.vortices import render
from examples.mechanics.vortices.core import Vortices, ga, grid

np.set_printoptions(precision=3, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Even = ga.gatype.even()
Gradient = ga.gatype((Vector, Vector))                                       # Vector <- Vector
plane = mv.xy

# An outer and an inner pair on one axis, each pair turning in opposite senses so that it travels
# along x, one entry per vortex.
half_spans = np.array([1.0, -1.0, 0.55, -0.55])
circulations = np.array([1.0, -1.0, 1.0, -1.0])
core_radius = 0.15
# The pattern does not repeat: its only copy is itself.
alone = mv.vector([[0.0, 0.0]])                                              # [copies] Vector
half_width, half_height = 2.4, 1.6
columns, rows = 300, 200
dt = 0.02
steps = 600
vorticity_limit, swirl_limit = 8.0, 6.0

## 1. The flow around the vortices

Each vortex turns the fluid around its centre in the plane `xy`. A separation `r` from the centre, turned a quarter turn in the plane, is `r | plane`, and the vortex moves the fluid along it. How fast follows the vortex blob of Rosenhead and Moore: near the centre the fluid turns like a solid disc, and beyond the core the speed falls off as one over the distance. The velocity at a point is the sum over the vortices.

In [ ]:
def separations(vortices: Vortices, points: Vector) -> Vector:
    """From every vortex and each of its copies to every point."""
    return points[..., None, None] - (vortices.centres[..., None] + vortices.copies)   # [..., vortices, copies] Vector


def profile(vortices: Vortices, separations: Vector) -> tuple[Scalar, Scalar]:
    """The rate at which the fluid turns about each centre, and its change with the squared distance."""
    blob = (separations | separations) + vortices.core_radius**2                  # [..., vortices, copies] Scalar
    rate = vortices.circulations[:, None] / (2 * np.pi) / blob                   # [..., vortices, copies] Scalar
    return rate, -rate / blob


def velocity(vortices: Vortices, points: Vector) -> Vector:
    offsets = separations(vortices, points)                                       # [..., vortices, copies] Vector
    rate, _ = profile(vortices, offsets)                                          # [..., vortices, copies] Scalar
    return (rate * (offsets | plane)).sum(axis=-1).sum(axis=-1)                   # [...] Vector


vortices = Vortices((mv.y * half_spans).cast(Vector), circulations, core_radius, alone)
# A window around the vortices, sampled at the centres of its cells.
points = vortices.centres.mean(axis=-1) + grid(half_width, half_height, columns, rows)   # [rows, columns] Vector
flow = velocity(vortices, points)                                                 # [rows, columns] Vector

## 2. The derivative of the flow

The velocity gradient at a point is a map on displacements, `Vector <- Vector`: the change in velocity over a small step. A step turns with the rate at the point, `rate * (Vector | plane)`, and it also changes the rate by its component along the separation, `separation | Vector`, which acts on the turned separation. Contracting the gradient against an open vector, `(Vector * gradient(Vector)).contract()`, pairs that vector with the gradient's input through the metric: this is the derivative of the flow. Its scalar part is the divergence, which vanishes here, as the fluid neither gathers nor spreads. Its bivector part is the vorticity, the rate at which the fluid turns, concentrated in the cores. Away from them the vorticity falls off as the fourth power of the distance and the whole derivative all but vanishes: there the flow nearly meets the Cauchy–Riemann condition, the mark of an analytic flow, which point vortices without cores would meet exactly.

In the notation of vector calculus the scalar part reads as $\nabla\cdot\mathbf v$ and the bivector part as $(\partial_x v_y - \partial_y v_x)\,\mathbf e_x\wedge\mathbf e_y$.

In the notation of complex analysis, with the velocity $\mathbf v = u\,\mathbf e_x + v\,\mathbf e_y$ written as $u - iv$, the vanishing derivative reads as the Cauchy–Riemann equations $\partial_x u + \partial_y v = 0$ and $\partial_y u - \partial_x v = 0$.

In [ ]:
def gradient(vortices: Vortices, points: Vector) -> Gradient:
    offsets = separations(vortices, points)                                       # [..., vortices, copies] Vector
    rate, rate_change = profile(vortices, offsets)                                # [..., vortices, copies] Scalar each
    turning = rate * (Vector | plane)                                             # [..., vortices, copies] Vector <- Vector
    spreading = 2 * rate_change * (offsets | plane) * (offsets | Vector)          # [..., vortices, copies] Vector <- Vector
    return (turning + spreading).sum(axis=-1).sum(axis=-1)                        # [...] Vector <- Vector


def derivative(gradients: Gradient) -> Even:
    return (Vector * gradients(Vector)).contract()                                # [...] Even


gradients = gradient(vortices, points)                                            # [rows, columns] Vector <- Vector
derivatives = derivative(gradients)                                               # [rows, columns] Even
# The point midway between the two vortices above the axis.
between = (vortices.centres[0] + vortices.centres[2]) / 2                         # [] Vector
between_gradient = gradient(vortices, between)                                    # [] Vector <- Vector
between_derivative = derivative(between_gradient)                                 # [] Even

Printed below, the gradient's coefficients at that point are not symmetric. The derivative there has no scalar part, and its vorticity is the difference of the two off-diagonal coefficients, the rate at which the fluid turns.

In [ ]:
print(f"gradient\n{between_gradient.coefficient_table()}\n")
print(f"derivative, {between_derivative.output_subspace}\n{between_derivative.kernel}")

## 3. Swirl against strain

The gradient's skew part turns the fluid and its symmetric part stretches it. The trace of the gradient applied twice pairs its output with its input and needs no metric, and it weighs the two against each other: `-0.5 * gradient(gradient).trace()` is positive where the fluid turns faster than it stretches, inside the cores, and negative where the pairs pull the fluid between them apart. For a flow without divergence in the plane it equals the gradient's determinant. Midway between the upper vortices it is negative: the mean of the off-diagonal coefficients, the stretch, outweighs the turn.

In the tensor notation of the velocity gradient the swirl reads as the Q-criterion, $Q = \tfrac12\left(\lVert\Omega\rVert^2 - \lVert S\rVert^2\right)$, with $\Omega$ and $S$ the skew and symmetric parts of the velocity gradient.

In [ ]:
def swirl(gradients: Gradient) -> Scalar:
    return -0.5 * gradients(gradients).trace()                                    # [...] Scalar


swirls = swirl(gradients)                                                         # [rows, columns] Scalar
between_swirl = swirl(between_gradient)                                           # [] Scalar
render.figure(vortices, points, derivatives, swirls, vorticity_limit, swirl_limit);

## 4. Leapfrogging

Each vortex moves with the flow at its centre, to which its own blob adds nothing, stepped by the classical fourth-order Runge–Kutta rule. The inner pair, narrower, travels faster and slips through the outer pair, which widens it and slows it down; the outer pair narrows, speeds up, and passes in its turn.

In [ ]:
def drift(vortices: Vortices) -> Vector:
    """The velocity of each vortex: the flow at its centre."""
    return velocity(vortices, vortices.centres)                                   # [vortices] Vector


def step(vortices: Vortices, dt: float) -> Vortices:
    def moved(rate: Vector, fraction: float) -> Vortices:
        return replace(vortices, centres=vortices.centres + rate * (fraction * dt))

    first = drift(vortices)                                                       # [vortices] Vector
    second = drift(moved(first, 0.5))                                             # [vortices] Vector
    third = drift(moved(second, 0.5))                                             # [vortices] Vector
    fourth = drift(moved(third, 1.0))                                             # [vortices] Vector
    return moved((first + 2 * second + 2 * third + fourth) / 6, 1.0)


travelled = vortices
for _ in range(steps):
    travelled = step(travelled, dt)
travelled_points = travelled.centres.mean(axis=-1) + grid(half_width, half_height, columns, rows)   # [rows, columns] Vector
travelled_gradients = gradient(travelled, travelled_points)                       # [rows, columns] Vector <- Vector
render.figure(travelled, travelled_points, derivative(travelled_gradients), swirl(travelled_gradients), vorticity_limit, swirl_limit);

In [ ]:
# checks
# The gradient is the change in velocity over a small displacement.
displacement = mv.vector([0.3, -0.2]) * 1e-5                                      # [] Vector
change = (velocity(vortices, points + displacement) - velocity(vortices, points - displacement)) / 2
np.testing.assert_allclose((gradients(displacement) - change).kernel, 0.0, atol=1e-12)
# The derivative has no divergence, and its bivector is the blobs' vorticity.
blob = (separations(vortices, points) | separations(vortices, points)) + core_radius**2   # [rows, columns, vortices, copies] Scalar
vorticity = (circulations[:, None] * core_radius**2 / np.pi / (blob * blob)).sum(axis=-1).sum(axis=-1)
np.testing.assert_allclose((derivatives - vorticity * plane).kernel, 0.0, atol=1e-10)
# Without divergence, swirl is the gradient's determinant, negative between the upper vortices.
np.testing.assert_allclose((swirls - gradients.det()).kernel, 0.0, atol=1e-10)
assert between_swirl.kernel[0] < 0